# Step 01_02_01 -- DuckDB Pre-Ingestion Investigation: sc2egset

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_02 -- EDA
**Dataset:** sc2egset
**Question:** How does DuckDB read_json_auto handle SC2EGSet replay JSONs?
What is the event array storage cost? What table split strategy is optimal?
**Invariants applied:** #6 (reproducibility), #9 (step scope)
**Step scope:** query
**Type:** Investigation only -- no full ingestion of 22,390 files

In [85]:
import json
import logging
import statistics
from pathlib import Path

import duckdb

from rts_predict.common.notebook_utils import get_reports_dir
from rts_predict.games.sc2.config import REPLAYS_SOURCE_DIR
from rts_predict.games.sc2.datasets.sc2egset.pre_ingestion import (
    select_sample_files,
    probe_read_json_auto_single,
    measure_event_arrays,
    probe_batch_ingestion,
    census_mapping_files,
    probe_mapping_read_json_auto,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## 1. Select sample files

In [86]:
inv_path = (
    get_reports_dir("sc2", "sc2egset")
    / "artifacts" / "01_exploration" / "01_acquisition"
    / "01_01_01_file_inventory.json"
)
with open(inv_path) as f:
    inventory = json.load(f)

samples = select_sample_files(inventory, REPLAYS_SOURCE_DIR)
print(f"Selected {len(samples)} sample files:")
for s in samples:
    mb = s.stat().st_size / 1024 / 1024
    print(f"  {s.parent.parent.name}/{s.name} ({mb:.1f} MB)")

Selected 7 sample files:
  2017_WCS_Austin/00cf6e6b68d835fae3a8f876b29a445c.SC2Replay.json (2.1 MB)
  2020_05_Dreamhack_Last_Chance/00690ef2aab0af33fcfe0b39fb7e073f.SC2Replay.json (14.3 MB)
  2020_TSL5/3f0f70a69cdb8c5160c8e3a41a137cc9.SC2Replay.json (143.1 MB)
  2019_WCS_Summer/88106fadcb47b78ac4c2dbdcbd8e5abc.SC2Replay.json (8.6 MB)
  2018_IEM_Katowice/8498226873709a61a6eab0af5534f581.SC2Replay.json (8.6 MB)
  2024_HomeStory_Cup_XXVI/8ba1f2a8ed7a9f5a88beaeeb50e1b242.SC2Replay.json (4.5 MB)
  2023_HomeStory_Cup_XXIV/7b2c35f81557f859096700ab653c49de.SC2Replay.json (3.3 MB)


## 1b. Filename uniqueness across tournaments

The replay files are named `<hash>.SC2Replay.json`. If the same hash appears
in multiple `*_data` directories, using just the basename as a provenance key
would be ambiguous. Check whether basenames are unique across all tournaments
before committing to a filename-only join key.

In [87]:
all_replays = sorted(REPLAYS_SOURCE_DIR.glob("*/*_data/*.SC2Replay.json"))
print(f"Total replay files: {len(all_replays):,}")

basenames = [f.name for f in all_replays]
unique_basenames = set(basenames)
print(f"Unique basenames:   {len(unique_basenames):,}")
print(f"Duplicates:         {len(basenames) - len(unique_basenames):,}")

if len(basenames) != len(unique_basenames):
    from collections import Counter
    dupes = [name for name, count in Counter(basenames).items() if count > 1]
    print(f"\nDuplicate basenames ({len(dupes)}):")
    for name in dupes[:10]:
        locations = [f.parent.parent.name for f in all_replays if f.name == name]
        print(f"  {name} -> {locations}")
    if len(dupes) > 10:
        print(f"  ... and {len(dupes) - 10} more")
else:
    print("\nAll basenames are unique — safe to use filename as join key.")

Total replay files: 22,390
Unique basenames:   22,390
Duplicates:         0

All basenames are unique — safe to use filename as join key.


## 2. Test read_json_auto on each sample

In [88]:
con = duckdb.connect(":memory:")
con.execute("SET memory_limit = '24GB'")
con.execute("SET threads = 4")

rja_results = []
for s in samples:
    r = probe_read_json_auto_single(con, s)
    rja_results.append(r)
    print(f"{r['file']}: success={r['success']}, cols={r.get('column_count')}")
    print(f"  ToonPlayerDescMap: {r.get('ToonPlayerDescMap_type', 'N/A')[:80]}...")

00cf6e6b68d835fae3a8f876b29a445c.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("98-S2-1-679" STRUCT(nickname VARCHAR, playerID BIGINT, userID BIGINT, SQ...
00690ef2aab0af33fcfe0b39fb7e073f.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("3-S2-1-3452013" STRUCT(nickname VARCHAR, playerID BIGINT, userID BIGINT,...
3f0f70a69cdb8c5160c8e3a41a137cc9.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("2-S2-1-1430346" STRUCT(nickname VARCHAR, playerID BIGINT, userID BIGINT,...
88106fadcb47b78ac4c2dbdcbd8e5abc.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("2-S2-1-8617761" STRUCT(nickname VARCHAR, playerID BIGINT, userID BIGINT,...
8498226873709a61a6eab0af5534f581.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("98-S2-1-997" STRUCT(nickname VARCHAR, playerID BIGINT, userID BIGINT, SQ...
8ba1f2a8ed7a9f5a88beaeeb50e1b242.SC2Replay.json: success=True, cols=11
  ToonPlayerDescMap: STRUCT("2-S2-1-1781626" STRU

## 3. DESCRIBE -- single file read_json_auto output

In [89]:
con.sql(
    f"DESCRIBE SELECT * FROM read_json_auto('{samples[0]}', "
    f"maximum_object_size=536870912)"
).show()

┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 4. Row preview -- single file

In [90]:
con.sql(
    f"SELECT * FROM read_json_auto('{samples[0]}', "
    f"maximum_object_size=536870912) LIMIT 5"
).show()

┌─────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 5. Event array storage assessment

In [91]:
ea_results = [measure_event_arrays(s) for s in samples]

for key in ("gameEvents", "trackerEvents", "messageEvents"):
    counts = [r[key]["element_count"] for r in ea_results]
    bytes_ = [r[key]["json_bytes"] for r in ea_results]
    total_est_gb = statistics.mean(bytes_) * 22390 / 1024**3
    print(f"{key}:")
    print(f"  elements: mean={statistics.mean(counts):.0f}, "
          f"median={statistics.median(counts):.0f}, "
          f"min={min(counts)}, max={max(counts)}")
    print(f"  est. total (22,390 files): {total_est_gb:.1f} GB")

gameEvents:
  elements: mean=79221, median=24825, min=6217, max=431109
  est. total (22,390 files): 326.8 GB
trackerEvents:
  elements: mean=5865, median=1745, min=521, max=33392
  est. total (22,390 files): 40.7 GB
messageEvents:
  elements: mean=34, median=1, min=1, max=228
  est. total (22,390 files): 0.1 GB


## 6. Batch ingestion probe

In [92]:
batch_dir = (
    REPLAYS_SOURCE_DIR / "2018_Cheeseadelphia_8"
    / "2018_Cheeseadelphia_8_data"
)
batch_result = probe_batch_ingestion(con, batch_dir)
print(f"Directory: {batch_result['directory']}")
print(f"Files: {batch_result['file_count']}")
print(f"Success: {batch_result['success']}")
print(f"Elapsed: {batch_result.get('elapsed_seconds', 'N/A')} seconds")
print(f"Rows: {batch_result.get('row_count', 'N/A')}")

Directory: 2018_Cheeseadelphia_8_data
Files: 64
Success: True
Elapsed: 1.58 seconds
Rows: 64


### DESCRIBE -- batch (union_by_name across tournament dir)

In [93]:
batch_glob = str(batch_dir / "*.SC2Replay.json")
con.sql(
    f"DESCRIBE SELECT * FROM read_json_auto('{batch_glob}', "
    f"union_by_name=true, filename=true, "
    f"maximum_object_size=536870912)"
).show()

┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### Row preview -- batch

In [94]:
con.sql(
    f"SELECT * FROM read_json_auto('{batch_glob}', "
    f"union_by_name=true, filename=true, "
    f"maximum_object_size=536870912) LIMIT 5"
).show()

┌─────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 6b. Event array struct analysis

The batch DESCRIBE (section 6) shows gameEvents, trackerEvents, messageEvents
as STRUCT[] — arrays of structs. But each event type within an array has a
different set of fields. DuckDB's read_json_auto unions all event structs
into one wide STRUCT with NULLs for missing fields, which is why the batch
DESCRIBE shows a single massive STRUCT type per array.

This section examines the actual event struct heterogeneity by reading
the raw JSON directly — how many distinct event struct shapes exist within
each array, and what are the common vs rare fields?

In [95]:
from collections import Counter
from rts_predict.common.json_utils import classify_value

EVENT_KEYS = ["gameEvents", "trackerEvents", "messageEvents"]

# Use 3 sample files spanning the size distribution
event_probe_files = [samples[0], samples[len(samples) // 2], samples[-1]]

for event_key in EVENT_KEYS:
    print(f"\n{'='*70}")
    print(f"  {event_key} — struct heterogeneity analysis")
    print(f"{'='*70}")

    type_counter: Counter[str] = Counter()
    type_shapes: dict[str, set[str]] = {}
    type_field_types: dict[str, dict[str, set[str]]] = {}
    total_events = 0

    for fp in event_probe_files:
        with fp.open() as fh:
            data = json.load(fh)
        events = data.get(event_key, [])
        total_events += len(events)

        for evt in events:
            if not isinstance(evt, dict):
                continue
            evt_type = evt.get("evtTypeName", evt.get("id", "UNKNOWN"))
            if isinstance(evt_type, int):
                evt_type = f"id={evt_type}"
            type_counter[evt_type] += 1

            if evt_type not in type_shapes:
                type_shapes[evt_type] = set()
                type_field_types[evt_type] = {}
            type_shapes[evt_type].update(evt.keys())

            for k, v in evt.items():
                tag = classify_value(v)
                if k not in type_field_types[evt_type]:
                    type_field_types[evt_type][k] = set()
                type_field_types[evt_type][k].add(tag)

    print(f"  Total events across {len(event_probe_files)} files: {total_events:,}")
    print(f"  Unique event types: {len(type_counter)}")

    all_fields: set[str] = set()
    for shape in type_shapes.values():
        all_fields.update(shape)
    print(f"  Union of all fields (what DuckDB STRUCT[] becomes): {len(all_fields)} fields")
    print()

    for evt_type, count in type_counter.most_common():
        shape = sorted(type_shapes[evt_type])
        nested = [
            f for f in shape
            if any("struct" in t or "list" in t
                   for t in type_field_types[evt_type].get(f, set()))
        ]
        print(f"  [{count:>7,}x] {evt_type}  ({len(shape)} fields)")
        if nested:
            print(f"           nested: {nested}")
            for nf in nested:
                print(f"             {nf}: {type_field_types[evt_type][nf]}")
        print()


  gameEvents — struct heterogeneity analysis
  Total events across 3 files: 42,893
  Unique event types: 11
  Union of all fields (what DuckDB STRUCT[] becomes): 41 fields

  [ 28,270x] CameraUpdate  (10 fields)
           nested: ['target', 'userid']
             target: {'null', 'struct(2 keys)'}
             userid: {'struct(1 keys)'}

  [  5,568x] ControlGroupUpdate  (7 fields)
           nested: ['mask', 'userid']
             mask: {'struct(1 keys)'}
             userid: {'struct(1 keys)'}

  [  2,657x] CommandManagerState  (6 fields)
           nested: ['userid']
             userid: {'struct(1 keys)'}

  [  1,971x] Cmd  (10 fields)
           nested: ['abil', 'cmdFlags', 'data', 'userid']
             abil: {'null', 'struct(3 keys)'}
             cmdFlags: {'list(str)'}
             data: {'struct(2 keys)', 'struct(1 keys)'}
             userid: {'struct(1 keys)'}

  [  1,880x] SelectionDelta  (6 fields)
           nested: ['delta', 'userid']
             delta: {'struct(4 key

### 6c. Event array — DuckDB STRUCT[] field explosion

When DuckDB reads heterogeneous event structs with union_by_name, it creates
one STRUCT type with ALL fields from ALL event types. This means each array
element carries NULLs for every field not present in its event type.

Show the actual DuckDB STRUCT[] column types to see how wide they become.

In [96]:
for event_key in EVENT_KEYS:
    print(f"\n{'='*60}")
    print(f"  typeof({event_key}) from batch load")
    print(f"{'='*60}")
    try:
        result = con.sql(f"""
            SELECT typeof("{event_key}") AS col_type
            FROM read_json_auto('{batch_glob}',
                union_by_name=true, maximum_object_size=536870912)
            LIMIT 1
        """).fetchone()
        type_str = result[0] if result else "N/A"
        # Print first 200 chars + total length to show the explosion
        print(f"  Length: {len(type_str)} chars")
        print(f"  Preview: {type_str[:300]}...")
    except Exception as e:
        print(f"  Error: {e}")


  typeof(gameEvents) from batch load
  Length: 2095 chars
  Preview: STRUCT(baseBuildNum BIGINT, buildNum BIGINT, cameraFollow BOOLEAN, debugPauseEnabled BOOLEAN, developmentCheatsEnabled BOOLEAN, evtTypeName VARCHAR, gameFullyDownloaded BOOLEAN, hotkeyProfile VARCHAR, id BIGINT, isMapToMapTransition BOOLEAN, loop BIGINT, multiplayerCheatsEnabled BOOLEAN, platformMac...

  typeof(trackerEvents) from batch load
  Length: 2121 chars
  Preview: STRUCT(evtTypeName VARCHAR, id BIGINT, loop BIGINT, playerId BIGINT, slotId BIGINT, "type" BIGINT, userId BIGINT, count BIGINT, upgradeTypeName VARCHAR, controlPlayerId BIGINT, creatorAbilityName VARCHAR, creatorUnitTagIndex BIGINT, creatorUnitTagRecycle BIGINT, unitTagIndex BIGINT, unitTagRecycle B...

  typeof(messageEvents) from batch load
  Length: 117 chars
  Preview: STRUCT(evtTypeName VARCHAR, id BIGINT, loop BIGINT, recipient BIGINT, string VARCHAR, userid STRUCT(userId BIGINT))[]...


### 6d. Alternative: UNNEST event arrays for per-event-type exploration

Instead of loading the wide STRUCT[], unnest the arrays and group by
evtTypeName to see the actual data shape per event type.

In [97]:
for event_key in EVENT_KEYS:
    print(f"\n{'='*60}")
    print(f"  UNNEST {event_key} — top event types with field counts")
    print(f"{'='*60}")
    try:
        con.sql(f"""
            WITH events AS (
                SELECT UNNEST("{event_key}") AS evt
                FROM read_json_auto('{batch_glob}',
                    union_by_name=true, maximum_object_size=536870912)
            )
            SELECT
                evt.evtTypeName AS event_type,
                COUNT(*) AS count
            FROM events
            GROUP BY evt.evtTypeName
            ORDER BY count DESC
        """).show()
    except Exception as e:
        print(f"  Error: {e}")


  UNNEST gameEvents — top event types with field counts
┌──────────────────────┬────────┐
│      event_type      │ count  │
│       varchar        │ int64  │
├──────────────────────┼────────┤
│ CameraUpdate         │ 610458 │
│ ControlGroupUpdate   │ 144806 │
│ CommandManagerState  │  91419 │
│ Cmd                  │  70593 │
│ SelectionDelta       │  69198 │
│ CmdUpdateTargetPoint │  61773 │
│ CmdUpdateTargetUnit  │  18094 │
│ CameraSave           │   1331 │
│ UserOptions          │    254 │
│ GameUserLeave        │    244 │
│ TriggerDialogControl │     32 │
│ UnitClick            │      2 │
│ TriggerPing          │      2 │
└──────────────────────┴────────┘
  13 rows             2 columns


  UNNEST trackerEvents — top event types with field counts
┌─────────────────┬───────┐
│   event_type    │ count │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ UnitBorn        │ 35019 │
│ UnitDied        │ 18182 │
│ PlayerStats     │ 11526 │
│ UnitTypeChange  │ 11199 │
│ UnitInit    

## 7. Mapping file census

In [98]:
census = census_mapping_files(REPLAYS_SOURCE_DIR)
print(f"Files found: {census['total_files_found']}")
print(f"Combined size: {census['total_combined_bytes']:,} bytes")
print(f"Cross-file consistency: {census['cross_file_consistency']}")

Files found: 70
Combined size: 4,284,224 bytes
Cross-file consistency: {'all_same_root_type': True, 'root_types': ['dict'], 'key_count_range': [1488, 1488], 'shared_keys_count': 1488, 'total_unique_keys': 1488, 'all_same_key_set': True}


In [99]:
first_mf = (
    REPLAYS_SOURCE_DIR / census["files"][0]["tournament"]
    / "map_foreign_to_english_mapping.json"
)
mapping_test = probe_mapping_read_json_auto(con, first_mf)
print(f"read_json_auto: success={mapping_test['success']}")
print(f"DuckDB type: {mapping_test.get('columns', [{}])[0].get('column_type')}")

read_json_auto: success=True
DuckDB type: MAP(VARCHAR, VARCHAR)


## 8. Ingestion readiness checks

### 8a. Sub-field DESCRIBE for metadata structs

The root-level DESCRIBE (section 3) shows `details`, `header`, `initData`,
`metadata` as STRUCT. We need the sub-field types to write correct DDL.

In [100]:
con = duckdb.connect(":memory:")
con.execute("SET memory_limit = '24GB'")

# Use the batch glob (64 files) for richer union_by_name coverage
batch_glob = str(batch_dir / "*.SC2Replay.json")

for struct_col in ("details", "header", "initData", "metadata"):
    print(f"\n{'='*60}")
    print(f"  DESCRIBE {struct_col}.*")
    print(f"{'='*60}")
    try:
        con.sql(f"""
            DESCRIBE SELECT "{struct_col}".* FROM read_json_auto(
                '{batch_glob}', union_by_name=true,
                maximum_object_size=536870912
            )
        """).show()
    except Exception as e:
        print(f"  Cannot unnest {struct_col}: {e}")
        con.sql(f"""
            SELECT typeof("{struct_col}") AS type
            FROM read_json_auto('{batch_glob}',
                union_by_name=true, maximum_object_size=536870912)
            LIMIT 1
        """).show()


  DESCRIBE details.*
┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ gameSpeed     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ isBlizzardMap │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ timeUTC       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘


  DESCRIBE header.*
┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ elapsedGameLoops │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ ver

### 8b. Storage estimate -- median vs mean (skew correction)

In [101]:
print("Storage estimates -- mean vs median (7-file sample):\n")
print(f"{'Array':<20} {'Mean (GB)':>10} {'Median (GB)':>12} {'Ratio':>8}")
print("-" * 55)
for key in ("gameEvents", "trackerEvents", "messageEvents"):
    bytes_ = [r[key]["json_bytes"] for r in ea_results]
    mean_gb = statistics.mean(bytes_) * 22390 / 1024**3
    median_gb = statistics.median(bytes_) * 22390 / 1024**3
    ratio = mean_gb / median_gb if median_gb > 0 else float("inf")
    print(f"{key:<20} {mean_gb:>10.1f} {median_gb:>12.1f} {ratio:>8.1f}x")

total_mean = sum(statistics.mean([r[k]["json_bytes"] for r in ea_results]) for k in ("gameEvents", "trackerEvents", "messageEvents")) * 22390 / 1024**3
total_median = sum(statistics.median([r[k]["json_bytes"] for r in ea_results]) for k in ("gameEvents", "trackerEvents", "messageEvents")) * 22390 / 1024**3
print(f"{'TOTAL':<20} {total_mean:>10.1f} {total_median:>12.1f} {total_mean/total_median:>8.1f}x")
print(f"\nNote: n=7, highly right-skewed. Median is the conservative estimate.")

Storage estimates -- mean vs median (7-file sample):

Array                 Mean (GB)  Median (GB)    Ratio
-------------------------------------------------------
gameEvents                326.8        107.7      3.0x
trackerEvents              40.7         12.2      3.3x
messageEvents               0.1          0.0     37.1x
TOTAL                     367.5        119.9      3.1x

Note: n=7, highly right-skewed. Median is the conservative estimate.


### 8c. ToonPlayerDescMap -- per-player fields catalog

Enumerate the actual per-player fields inside ToonPlayerDescMap to assess
whether these are prediction-relevant (race, MMR, APM, etc.)

In [102]:
with open(samples[0]) as f:
    sample_data = json.load(f)

tpdm = sample_data.get("ToonPlayerDescMap", {})
if tpdm:
    first_player_key = next(iter(tpdm))
    player_data = tpdm[first_player_key]
    print(f"ToonPlayerDescMap: {len(tpdm)} players in sample file")
    print(f"Per-player fields ({len(player_data)} total):")
    for k, v in sorted(player_data.items()):
        print(f"  {k}: {type(v).__name__} = {repr(v)[:80]}")

ToonPlayerDescMap: 2 players in sample file
Per-player fields (20 total):
  APM: int = 292
  MMR: int = 0
  SQ: int = 103
  clanTag: str = ''
  color: dict = {'a': 255, 'b': 0, 'g': 66, 'r': 255}
  handicap: int = 100
  highestLeague: str = 'Unknown'
  isInClan: bool = False
  nickname: str = 'Has'
  playerID: int = 2
  race: str = 'Prot'
  realm: str = 'Unknown'
  region: str = 'Unknown'
  result: str = 'Loss'
  selectedRace: str = 'Prot'
  startDir: int = 5
  startLocX: int = 161
  startLocY: int = 21
  supplyCappedPercent: int = 8
  userID: int = 1


### 8d. Smoke test — extract_events_to_parquet on a single replay

Validate that the Parquet extraction pipeline works end-to-end on one file
before committing to the full 22,390-file run. Uses a temp raw directory
with a single replay, then verifies the output with in-memory DuckDB.

In [103]:
import shutil
import tempfile

from rts_predict.games.sc2.config import DUCKDB_TEMP_DIR
from rts_predict.games.sc2.datasets.sc2egset.ingestion import extract_events_to_parquet

sample = samples[0]
smoke_output = DUCKDB_TEMP_DIR / "events_smoke_test"

# Build a temp raw dir matching the glob pattern: */*_data/*.SC2Replay.json
with tempfile.TemporaryDirectory() as tmp_raw:
    mirror = Path(tmp_raw) / sample.parent.parent.name / sample.parent.name
    mirror.mkdir(parents=True)
    shutil.copy2(sample, mirror / sample.name)

    counts = extract_events_to_parquet(Path(tmp_raw), smoke_output)

print("Extraction results:")
for event_type, count in counts.items():
    print(f"  {event_type}: {count:,} rows")

INFO:rts_predict.games.sc2.datasets.sc2egset.ingestion:extract_events_to_parquet: found 1 replay files
INFO:rts_predict.games.sc2.datasets.sc2egset.ingestion:gameEvents: 6217 total rows -> /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/tmp/events_smoke_test/gameEvents.parquet
INFO:rts_predict.games.sc2.datasets.sc2egset.ingestion:trackerEvents: 521 total rows -> /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/tmp/events_smoke_test/trackerEvents.parquet
INFO:rts_predict.games.sc2.datasets.sc2egset.ingestion:messageEvents: 1 total rows -> /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/sc2/datasets/sc2egset/data/tmp/events_smoke_test/messageEvents.parquet


Extraction results:
  gameEvents: 6,217 rows
  trackerEvents: 521 rows
  messageEvents: 1 rows


In [104]:
# Verify Parquet output with in-memory DuckDB
con_smoke = duckdb.connect(":memory:")

for event_type in ("gameEvents", "trackerEvents", "messageEvents"):
    pq_path = smoke_output / f"{event_type}.parquet"
    if not pq_path.exists():
        print(f"{event_type}: no Parquet file (0 events)")
        continue

    size_kb = pq_path.stat().st_size / 1024
    print(f"\n{'='*60}")
    print(f"  {event_type}.parquet ({size_kb:.1f} KB)")
    print(f"{'='*60}")

    con_smoke.sql(f"DESCRIBE SELECT * FROM '{pq_path}'").show()

    row_count = con_smoke.sql(f"SELECT COUNT(*) FROM '{pq_path}'").fetchone()[0]
    print(f"  Rows: {row_count:,} (expected: {counts[event_type]:,})")
    assert row_count == counts[event_type], "Row count mismatch!"

    nulls = con_smoke.sql(f"""
        SELECT
            SUM(CASE WHEN filename IS NULL THEN 1 ELSE 0 END) AS null_filename,
            SUM(CASE WHEN loop IS NULL THEN 1 ELSE 0 END) AS null_loop,
            SUM(CASE WHEN "evtTypeName" IS NULL THEN 1 ELSE 0 END) AS null_evt_type
        FROM '{pq_path}'
    """).fetchone()
    print(f"  NULLs: filename={nulls[0]}, loop={nulls[1]}, evtTypeName={nulls[2]}")

    con_smoke.sql(f"""
        SELECT "evtTypeName", COUNT(*) AS n
        FROM '{pq_path}'
        GROUP BY "evtTypeName"
        ORDER BY n DESC
    """).show()

con_smoke.close()


  gameEvents.parquet (117.5 KB)
┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ filename    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ loop        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ evtTypeName │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ event_data  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

  Rows: 6,217 (expected: 6,217)
  NULLs: filename=0, loop=0, evtTypeName=0
┌──────────────────────┬───────┐
│     evtTypeName      │   n   │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ CameraUpdate         │  2323 │
│ ControlGroupUpdate   │  1844 │
│ CommandManagerState  │   618 │
│ SelectionDelta       │   440

In [105]:
# Compression ratio
raw_bytes = sample.stat().st_size
parquet_bytes = sum(
    (smoke_output / f"{et}.parquet").stat().st_size
    for et in ("gameEvents", "trackerEvents", "messageEvents")
    if (smoke_output / f"{et}.parquet").exists()
)
print(f"Raw JSON:      {raw_bytes / 1024:.1f} KB")
print(f"Parquet total: {parquet_bytes / 1024:.1f} KB")
print(f"Compression:   {raw_bytes / parquet_bytes:.1f}x")

shutil.rmtree(smoke_output)
print("\nSmoke test passed — output cleaned up.")

Raw JSON:      2162.2 KB
Parquet total: 135.7 KB
Compression:   15.9x

Smoke test passed — output cleaned up.


## 9. Findings and ingestion strategy recommendation

Summarize all findings from sections 1-8 here after execution.
Key decisions to record:

- **Table split strategy:** metadata-only vs include events?
- **Event ingestion:** defer all, or ingest specific trackerEvent types?
- **ToonPlayerDescMap:** unnest to per-player columns or keep as MAP?
- **Mapping files:** single reference table (all 70 identical)?
- **Sub-field types:** any surprises in nested struct fields?
- **Raw layer strategy:** `SELECT *` with `filename=true` for all `*_raw`
  tables — no explicit DDL at this stage. DDL is deferred to staging tables
  after exploration, analysis, and cleaning.

In [106]:
con.close()